# Parquet file data cleaning

In [35]:
import pandas as pd
import numpy as np
import os
import sys

PARQUET_PATH = "../parquet_files/"
OUTPUT_PATH = os.path.join(PARQUET_PATH, "cleaned_files/")

In [2]:
positive_parq = pd.read_parquet(os.path.join(PARQUET_PATH, "positive.parquet"))
negative_parq = pd.read_parquet(os.path.join(PARQUET_PATH, "negative.parquet"))

In [3]:
# Configuration 
missing_threshold = 0.2
min_num_rows = 100
formant_cols = ["F0", "F1", "F2", "F3"]

start_time = 0.02
start_tol = 0.001                  
max_frames = 1350             
row_rule = "any"  

In [4]:
def add_series_ids(df):
    df = df.sort_values(["healthCode"], kind="mergesort").copy()

    series_ids = []
    prev_hc = None
    prev_time = None
    series_id = -1
    frame_in_series = 0
    
    for hc, t in zip(df["healthCode"].to_numpy(), df["time"].to_numpy()):
        # new healthCode => new series
        new_user = (hc != prev_hc)
    
        # time reset (goes backward OR is at start time)
        if (not new_user) and (prev_time is not None):
            time_reset = (t < prev_time) or np.isclose(t, start_time, atol=start_tol)
        else:
            time_reset = False
    
        # cap at max_frames
        over_cap = (frame_in_series >= max_frames)
    
        # Start a new series if any rule triggers
        if new_user or time_reset or over_cap:
            series_id += 1
            frame_in_series = 0
    
        # Record series id for this row
        series_ids.append(series_id)
    
        # Update trackers
        frame_in_series += 1
        prev_hc = hc
        prev_time = t
    
    df["series_id"] = series_ids
    return df

def mark_nans(df):
    if row_rule == "all":
        df["is_nan"] = df[formant_cols].isna().all(axis=1)
    else:
        df["is_nan"] = df[formant_cols].isna().any(axis=1)
    return df

In [5]:
# Steps below are separated to sanity check. 
positive_parq_series = add_series_ids(positive_parq)
negative_parq_series = add_series_ids(negative_parq)

In [6]:
pos_no_nan = mark_nans(positive_parq_series)
neg_no_nan = mark_nans(negative_parq_series)

In [7]:
pos_no_nan["series_id"].unique()

array([    0,     1,     2, ..., 39374, 39375, 39376], shape=(39377,))

In [8]:
neg_no_nan["series_id"].unique()

array([    0,     1,     2, ..., 23423, 23424, 23425], shape=(23426,))

In [9]:
pos_summary = (
    pos_no_nan.groupby(["healthCode","series_id"], sort=False)
      .agg(rows=("time","size"), nan_rows=("is_nan","sum"))
      .reset_index()
)
pos_summary["nan_frac"] = pos_summary["nan_rows"] / pos_summary["rows"]
pos_summary.head()

,healthCode,series_id,rows,nan_rows,nan_frac
0,0085ab2b-7d74-4e88-8117-e9259adb6266,0,998,195,0.195391
1,0085b356-0550-4cf1-85bd-2bcd89bf1201,1,1004,30,0.029880
2,0085b356-0550-4cf1-85bd-2bcd89bf1201,2,1002,176,0.175649
3,00dc061b-8151-44cc-8eae-4d10f11a5ab6,3,1001,5,0.004995
4,00dc061b-8151-44cc-8eae-4d10f11a5ab6,4,1000,1,0.001000


In [10]:
neg_summary = (
    neg_no_nan.groupby(["healthCode","series_id"], sort=False)
      .agg(rows=("time","size"), nan_rows=("is_nan","sum"))
      .reset_index()
)
neg_summary["nan_frac"] = neg_summary["nan_rows"] / neg_summary["rows"]
neg_summary.head()

,healthCode,series_id,rows,nan_rows,nan_frac
0,000240d1-1110-4dd2-a2d0-e344c37efd68,0,1003,319,0.318046
1,00081bd9-9abd-4003-b035-de6cc3e8c922,1,1002,425,0.424152
2,00290381-e82e-46b2-b4e6-df115823d71b,2,1000,94,0.094000
3,00547584-0c04-4228-a5d5-c68f7d59f176,3,1000,9,0.009000
4,00547584-0c04-4228-a5d5-c68f7d59f176,4,998,48,0.048096


In [11]:
bad_pos_series = pos_summary.query("nan_frac > @missing_threshold or rows < @min_num_rows")
bad_pos_keys = set(zip(bad_pos_series["healthCode"], bad_pos_series["series_id"]))

print(f"Found {len(bad_pos_keys)} bad recordings with a positive diagnosis")

Found 5159 bad recordings with a positive diagnosis


In [12]:
bad_neg_series = neg_summary.query("nan_frac > @missing_threshold or rows < @min_num_rows")
bad_neg_keys = set(zip(bad_neg_series["healthCode"], bad_neg_series["series_id"]))

print(f"Found {len(bad_neg_keys)} bad recordings with a negative diagnosis")

Found 2299 bad recordings with a negative diagnosis


In [13]:
pos_mask = ~pos_no_nan[["healthCode","series_id"]].apply(tuple, axis=1).isin(bad_pos_keys)
clean_pos_df = pos_no_nan[pos_mask].copy()

print(f"Original rows: {len(pos_no_nan)} | Clean rows: {len(clean_pos_df)}")

Original rows: 39379394 | Clean rows: 34224132


In [14]:
neg_mask = ~neg_no_nan[["healthCode","series_id"]].apply(tuple, axis=1).isin(bad_neg_keys)
clean_neg_df = neg_no_nan[neg_mask].copy()

print(f"Original rows: {len(neg_no_nan)} | Clean rows: {len(clean_neg_df)}")

Original rows: 23433247 | Clean rows: 21136140


# Saving to files

In [32]:
clean_pos_df = clean_pos_df.drop(columns=["is_nan"])
clean_neg_df = clean_neg_df.drop(columns=["is_nan"])

In [33]:
clean_pos_df.head()

,time,F0,F1,F2,F3,healthCode,series_id
18971596,0.020748,NaN,NaN,NaN,NaN,0085ab2b-7d74-4e88-8117-e9259adb6266,0
18971597,0.030748,NaN,777.018552,2156.808878,2826.062931,0085ab2b-7d74-4e88-8117-e9259adb6266,0
18971598,0.040748,NaN,862.796612,1956.579605,2670.085505,0085ab2b-7d74-4e88-8117-e9259adb6266,0
18971599,0.050748,NaN,1240.188179,2235.394197,3275.075305,0085ab2b-7d74-4e88-8117-e9259adb6266,0
18971600,0.060748,NaN,1266.645755,2414.403012,3381.478874,0085ab2b-7d74-4e88-8117-e9259adb6266,0


In [34]:
clean_neg_df.head()

,time,F0,F1,F2,F3,healthCode,series_id
11009682,0.020317,NaN,770.622630,1898.053856,2891.707682,00290381-e82e-46b2-b4e6-df115823d71b,2
11009683,0.030317,NaN,765.209680,1822.178742,2923.549051,00290381-e82e-46b2-b4e6-df115823d71b,2
11009684,0.040317,NaN,696.819614,1687.041262,3104.636544,00290381-e82e-46b2-b4e6-df115823d71b,2
11009685,0.050317,NaN,741.941298,1654.518431,3169.142948,00290381-e82e-46b2-b4e6-df115823d71b,2
11009686,0.060317,NaN,810.052711,1698.202638,2829.603857,00290381-e82e-46b2-b4e6-df115823d71b,2


In [36]:
clean_pos_df.to_parquet(os.path.join(OUTPUT_PATH, "positive_clean.parquet"), index=False)
clean_neg_df.to_parquet(os.path.join(OUTPUT_PATH, "negative_clean.parquet"), index=False)

# Sanity check cleaned parquet files

In [40]:
pos_clean = pd.read_parquet(os.path.join(OUTPUT_PATH, "positive_clean.parquet"))
pos_clean.head()

,time,F0,F1,F2,F3,healthCode,series_id
0,0.020748,NaN,NaN,NaN,NaN,0085ab2b-7d74-4e88-8117-e9259adb6266,0
1,0.030748,NaN,777.018552,2156.808878,2826.062931,0085ab2b-7d74-4e88-8117-e9259adb6266,0
2,0.040748,NaN,862.796612,1956.579605,2670.085505,0085ab2b-7d74-4e88-8117-e9259adb6266,0
3,0.050748,NaN,1240.188179,2235.394197,3275.075305,0085ab2b-7d74-4e88-8117-e9259adb6266,0
4,0.060748,NaN,1266.645755,2414.403012,3381.478874,0085ab2b-7d74-4e88-8117-e9259adb6266,0


In [41]:
neg_clean = pd.read_parquet(os.path.join(OUTPUT_PATH, "negative_clean.parquet"))
neg_clean.head()

,time,F0,F1,F2,F3,healthCode,series_id
0,0.020317,NaN,770.622630,1898.053856,2891.707682,00290381-e82e-46b2-b4e6-df115823d71b,2
1,0.030317,NaN,765.209680,1822.178742,2923.549051,00290381-e82e-46b2-b4e6-df115823d71b,2
2,0.040317,NaN,696.819614,1687.041262,3104.636544,00290381-e82e-46b2-b4e6-df115823d71b,2
3,0.050317,NaN,741.941298,1654.518431,3169.142948,00290381-e82e-46b2-b4e6-df115823d71b,2
4,0.060317,NaN,810.052711,1698.202638,2829.603857,00290381-e82e-46b2-b4e6-df115823d71b,2
